# CSE 251B FineWeb-Edu Pipeline

This notebook prepares FineWeb-Edu shards, trains the nano model, runs the TA evaluation script, and packages the submission files under `results/<timestamp>` before copying them to Drive.

In [3]:
!nvidia-smi

Tue May 26 03:25:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             48W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. Mount Drive And Paths

In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import shutil
import subprocess
import time

DRIVE_ROOT = Path('/content/drive/MyDrive/251B')
DRIVE_DATASET_DIR = DRIVE_ROOT / 'dataset' / 'fineweb_edu'
DRIVE_RESULTS_DIR = DRIVE_ROOT / 'results'

REPO_DIR = Path('/content/milestone')
LOCAL_DATASET_DIR = REPO_DIR / 'data' / 'fineweb_edu'

RUN_NAME = 'fineweb_edu_nano_' + time.strftime('%Y%m%d_%H%M%S')
LOCAL_RESULTS_DIR = REPO_DIR / 'results' / RUN_NAME
DRIVE_RUN_DIR = DRIVE_RESULTS_DIR / RUN_NAME

DRIVE_DATASET_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Drive dataset:', DRIVE_DATASET_DIR)
print('Drive results:', DRIVE_RESULTS_DIR)
print('Repo:', REPO_DIR)
print('Local dataset:', LOCAL_DATASET_DIR)
print('Local run results:', LOCAL_RESULTS_DIR)
print('Drive run results:', DRIVE_RUN_DIR)

Mounted at /content/drive
Drive dataset: /content/drive/MyDrive/251B/dataset/fineweb_edu
Drive results: /content/drive/MyDrive/251B/results
Repo: /content/milestone
Local dataset: /content/milestone/data/fineweb_edu
Local run results: /content/milestone/results/fineweb_edu_nano_20260526_032253
Drive run results: /content/drive/MyDrive/251B/results/fineweb_edu_nano_20260526_032253


## 2. Prepare Repository

In [4]:
!rm -rf /content/milestone
!git clone -b muon https://github.com/FufenNan/milestone.git /content/milestone
%cd /content/milestone
!git status --short --branch

LOCAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

Cloning into '/content/milestone'...
remote: Enumerating objects: 798, done.
remote: Counting objects: 100% (798/798), done.
remote: Compressing objects: 100% (369/369), done.
remote: Total 798 (delta 446), reused 763 (delta 422), pack-reused 0 (from 0)
Receiving objects: 100% (798/798), 7.73 MiB | 20.10 MiB/s, done.
Resolving deltas: 100% (446/446), done.
/content/milestone
## muon...origin/muon


## 3. Install Dependencies

In [5]:
!pip install -q datasets tiktoken tqdm

## 4. Copy Or Prepare FineWeb-Edu

In [6]:
DATASET_LOG = LOCAL_RESULTS_DIR / 'dataset_prepare.log'

def log(msg):
    line = f'[{time.strftime("%Y-%m-%d %H:%M:%S")}] {msg}'
    print(line)
    with open(DATASET_LOG, 'a', encoding='utf-8') as f:
        f.write(line + '\n')

def count_shards(path):
    path = Path(path)
    train = sorted(path.glob('fineweb_train_*.bin'))
    val = sorted(path.glob('fineweb_val_*.bin'))
    return len(train), len(val)

def shard_bytes(path):
    path = Path(path)
    return sum(p.stat().st_size for p in path.glob('fineweb_*.bin'))

def has_ready_dataset(path, min_train_shards=1):
    train_count, val_count = count_shards(path)
    return train_count >= min_train_shards and val_count >= 1

def sync_dir(src, dst):
    src = Path(src)
    dst = Path(dst)
    dst.mkdir(parents=True, exist_ok=True)
    try:
        subprocess.run(['rsync', '-ah', '--info=progress2', f'{src}/', f'{dst}/'], check=True)
    except FileNotFoundError:
        shutil.copytree(src, dst, dirs_exist_ok=True)

SHARD_SIZE = 100_000_000
MAX_SHARDS = 22
MIN_TRAIN_SHARDS = MAX_SHARDS - 1
NUM_PROC = max(1, (os.cpu_count() or 2) // 2)

drive_train, drive_val = count_shards(DRIVE_DATASET_DIR)
log(f'Drive shards: train={drive_train}, val={drive_val}, size={shard_bytes(DRIVE_DATASET_DIR) / 1e9:.2f} GB')

if has_ready_dataset(DRIVE_DATASET_DIR, MIN_TRAIN_SHARDS):
    log('Copying FineWeb-Edu from Drive to repo data/fineweb_edu...')
    sync_dir(DRIVE_DATASET_DIR, LOCAL_DATASET_DIR)
else:
    log('Drive dataset is incomplete. Preparing FineWeb-Edu under repo data/fineweb_edu...')
    LOCAL_DATASET_DIR.mkdir(parents=True, exist_ok=True)
    cmd = [
        'python', '-u', 'data/fineweb.py',
        '--data-root', str(LOCAL_DATASET_DIR),
        '--streaming',
        '--shard-size', str(SHARD_SIZE),
        '--max-shards', str(MAX_SHARDS),
        '--num-proc', str(NUM_PROC),
    ]
    log('Running: ' + ' '.join(cmd))
    with open(DATASET_LOG, 'a', encoding='utf-8') as log_file:
        proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
        ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f'Dataset preparation failed with exit code {ret}. See {DATASET_LOG}')
    log('Copying prepared FineWeb-Edu shards back to Drive...')
    sync_dir(LOCAL_DATASET_DIR, DRIVE_DATASET_DIR)

local_train, local_val = count_shards(LOCAL_DATASET_DIR)
log(f'Local shards: train={local_train}, val={local_val}, size={shard_bytes(LOCAL_DATASET_DIR) / 1e9:.2f} GB')
assert has_ready_dataset(LOCAL_DATASET_DIR, 1), 'FineWeb-Edu dataset is not ready in repo data/fineweb_edu.'

[2026-05-26 03:26:22] Drive shards: train=21, val=1, size=4.40 GB
[2026-05-26 03:26:22] Copying FineWeb-Edu from Drive to repo data/fineweb_edu...
[2026-05-26 03:30:04] Local shards: train=21, val=1, size=4.40 GB


## 5. Train

In [7]:
TRAIN_STDOUT = LOCAL_RESULTS_DIR / 'train_stdout.log'

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

import importlib.util
spec = importlib.util.spec_from_file_location('train_config', REPO_DIR / 'config' / 'config.py')
train_config = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train_config)
print('Optimizer:', getattr(train_config, 'optimizer', 'adamw'))
print('Muon lr:', getattr(train_config, 'muon_lr', None))

cmd = ['python', '-u', 'train.py', '--config', 'config/config.py']
# cmd = ['python', '-u', 'train.py', '--config', 'config/config_10000.py']
print('Run name:', RUN_NAME)
print('Command:', ' '.join(cmd))

with open(TRAIN_STDOUT, 'w', encoding='utf-8') as log_file:
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
        log_file.write(line)
        log_file.flush()
    ret = proc.wait()

if ret != 0:
    raise RuntimeError(f'Training failed with exit code {ret}. See {TRAIN_STDOUT}')

Optimizer: muon
Muon lr: 0.02
Run name: fineweb_edu_nano_20260526_032253
Command: python -u train.py --config config/config.py
device: cuda
gradient accumulation steps: 33
parameters: 99,027,200
optimizer: muon | muon tensors: 60 (66,846,720 params) | adamw fallback tensors: 26 (32,180,480 params)
/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:2156: UserWarning: Torchinductor does not support code generation for complex operators. Performance may be worse than eager.
  warnings.warn(
step     0 | val loss 10.9502
step     0 | train loss 10.955290 | lr 7.5000e-07 | norm 8.5393 | 8608 tok/s
step    10 | train loss 10.624088 | lr 8.2500e-06 | norm 7.6464 | 179072 tok/s
step    20 | train loss 10.035824 | lr 1.5750e-05 | norm 5.3299 | 178724 tok/s
step    30 | train loss 9.539062 | lr 2.3250e-05 | norm 3.2554 | 178337 tok/s
step    40 | train loss 9.021864 | lr 3.0750e-05 | norm 3.7685 | 178304 tok/s
step    50 | train loss 8.611014 | lr 3.8250e-05 | norm 2.8261 | 1783

## 6. Run TA Evaluation

In [8]:
LOCAL_CHECKPOINT = REPO_DIR / 'checkpoints' / 'checkpoint.pt'
EVAL_DATA = REPO_DIR / 'val.bin'
TA_EVAL_JSON = LOCAL_RESULTS_DIR / 'eval_val.json'

assert LOCAL_CHECKPOINT.exists(), f'Missing checkpoint: {LOCAL_CHECKPOINT}'
assert EVAL_DATA.exists(), f'Missing eval data: {EVAL_DATA}'

cmd = [
    'python', '-u', 'evaluate.py',
    '--model_dir', str(REPO_DIR),
    '--checkpoint_filename', 'checkpoints/checkpoint.pt',
    '--data', str(EVAL_DATA),
    '--device', 'cuda',
    '--batch_size', '1',
    '--output_json', str(TA_EVAL_JSON),
]
print('Command:', ' '.join(cmd))

proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
ret = proc.wait()

if ret != 0:
    raise RuntimeError(f'TA evaluation failed with exit code {ret}.')

Command: python -u evaluate.py --model_dir /content/milestone --checkpoint_filename checkpoints/checkpoint.pt --data /content/milestone/val.bin --device cuda --batch_size 1 --output_json /content/milestone/results/fineweb_edu_nano_20260526_032253/eval_val.json

--- Evaluating local model: /content/milestone ---
Loading model from /content/milestone/checkpoints/checkpoint.pt...
Total parameters:       99,027,200
Trainable parameters:   99,027,200
Evaluating on /content/milestone/val.bin (block_size=1024)...

  Perplexity:          26.8437
  Avg loss (nats):     3.290029
  Tokens evaluated:    5,170,176
  Total parameters:    99,027,200
  Eval time:           84.0s

Results written to /content/milestone/results/fineweb_edu_nano_20260526_032253/eval_val.json


## 7. Package Results And Copy To Drive

In [9]:
submission_files = {
    REPO_DIR / 'checkpoints' / 'checkpoint.pt': LOCAL_RESULTS_DIR / 'checkpoint.pt',
    REPO_DIR / 'config' / 'config.py': LOCAL_RESULTS_DIR / 'config.py',
    # REPO_DIR / 'config' / 'config_10000.py': LOCAL_RESULTS_DIR / 'config.py',
    REPO_DIR / 'model.py': LOCAL_RESULTS_DIR / 'model.py',
}

optional_logs = [
    REPO_DIR / 'checkpoints' / 'train.log',
    REPO_DIR / 'results' / 'train.log',
    REPO_DIR / 'checkpoints' / 'checkpoint_metadata.pt',
]

for src, dst in submission_files.items():
    assert src.exists(), f'Missing expected file: {src}'
    shutil.copy2(src, dst)

for src in optional_logs:
    if src.exists():
        shutil.copy2(src, LOCAL_RESULTS_DIR / src.name)

DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
sync_dir(LOCAL_RESULTS_DIR, DRIVE_RUN_DIR)

print('Local results:', LOCAL_RESULTS_DIR)
print('Drive results:', DRIVE_RUN_DIR)
print('Packaged files:')
for path in sorted(LOCAL_RESULTS_DIR.iterdir()):
    print(' -', path.name)

Local results: /content/milestone/results/fineweb_edu_nano_20260526_032253
Drive results: /content/drive/MyDrive/251B/results/fineweb_edu_nano_20260526_032253
Packaged files:
 - checkpoint.pt
 - checkpoint_metadata.pt
 - config.py
 - dataset_prepare.log
 - eval_val.json
 - model.py
 - train.log
 - train_stdout.log


# 8. Exit Colab

In [9]:
from google.colab import runtime
runtime.unassign()